# 도구 정의와 첫 호출 토큰

MCP 서버에서 발견한 도구 정의의 수가 늘면 같은 첫 모델 호출의 입력 토큰은 얼마나 달라질까요?

## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 순서대로 실행합니다. 비밀 값은 출력하지 않습니다.

## 패키지와 API 키 확인

패키지와 API 키 확인에 필요한 값과 결과를 확인합니다.

In [1]:
# 패키지와 API 키 확인에 필요한 값을 준비하고 결과를 확인합니다.
%pip install -q "langchain[mcp]==1.4.0" langchain-openai==1.6.0 fastmcp==4.0.3 mcp==2.2.0 python-dotenv==1.2.3
import warnings
from importlib.metadata import version
from langchain_core._api import LangChainBetaWarning

warnings.filterwarnings("ignore", message=r".*langchain\.mcp.*", category=LangChainBetaWarning)

for package in ["langchain", "langchain-openai", "fastmcp", "mcp"]:
    print(package, version(package))
import os
from pathlib import Path
from dotenv import load_dotenv

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
print("API 키 존재 여부 확인 완료")

Note: you may need to restart the kernel to use updated packages.


langchain 1.4.0
langchain-openai 1.6.0
fastmcp 4.0.3
mcp 2.2.0
API 키 존재 여부 확인 완료


## 직접 도구와 소형 MCP 서버 준비

직접 도구와 소형 MCP 서버 준비에 필요한 값과 결과를 확인합니다.

In [2]:
# 직접 도구와 소형 MCP 서버 준비에 필요한 값을 준비하고 결과를 확인합니다.
import asyncio
from fastmcp import FastMCP
from langchain.mcp import MCPAdapter
from langchain.tools import tool

@tool
def direct_lookup(topic: str) -> str:
    """자료실 규정을 조회합니다. topic은 대출, 운영, 연체 중 하나입니다."""
    return topic

small = FastMCP("small")
@small.tool
def lookup_policy(topic: str) -> str:
    """자료실 규정을 조회합니다. topic은 대출, 운영, 연체 중 하나입니다."""
    return topic

async def load_mcp_tools(server: object) -> list:
    async with MCPAdapter(server) as adapter:
        return await adapter.list_tools()

small_tools = await load_mcp_tools(small)
print("small count:", len(small_tools))
assert len(small_tools) == 1

small count: 1


## 대형 MCP 서버와 다섯 도구 준비

대형 MCP 서버와 다섯 도구 준비에 필요한 값과 결과를 확인합니다.

In [3]:
# 대형 MCP 서버와 다섯 도구 준비에 필요한 값을 준비하고 결과를 확인합니다.
large = FastMCP("large")
for name, description in [
    ("lookup_policy", "자료실 규정을 조회합니다."),
    ("search_catalog", "도서 제목을 검색합니다."),
    ("check_hours", "운영 시간을 확인합니다."),
    ("list_events", "공개 행사를 조회합니다."),
    ("find_room", "자료실 위치를 조회합니다."),
]:
    async def read_only(query: str, _name: str = name) -> str:
        return f"{_name}: {query}"
    read_only.__name__ = name
    read_only.__doc__ = description + " query는 검색어입니다."
    large.tool(read_only, name=name)

large_tools = await load_mcp_tools(large)
print("counts:", 1, len(small_tools), len(large_tools))
assert len(small_tools) == 1 and len(large_tools) == 5

counts: 1 1 5


## 도구 정의 직렬화 크기 비교

도구 정의 직렬화 크기 비교에 필요한 값과 결과를 확인합니다.

In [4]:
# 도구 정의 직렬화 크기 비교에 필요한 값을 준비하고 결과를 확인합니다.
import json

def schema_chars(tools: list) -> int:
    payload = [{
        "name": item.name,
        "description": item.description,
        "schema": item.args_schema if isinstance(item.args_schema, dict) else item.args_schema.model_json_schema(),
    } for item in tools]
    return len(json.dumps(payload, ensure_ascii=False, sort_keys=True))

schema_sizes = {
    "direct_1": schema_chars([direct_lookup]),
    "mcp_1": schema_chars(small_tools),
    "mcp_5": schema_chars(large_tools),
}
print("schema_chars:", schema_sizes)
assert schema_sizes["mcp_5"] > schema_sizes["mcp_1"]

schema_chars: {'direct_1': 289, 'mcp_1': 216, 'mcp_5': 1288}


## 같은 첫 모델 호출의 토큰 실측

같은 첫 모델 호출의 토큰 실측에 필요한 값과 결과를 확인합니다.

In [5]:
# 같은 첫 모델 호출의 토큰 실측에 필요한 값을 준비하고 결과를 확인합니다.
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-5.4-mini", reasoning_effort="none")
question = "자료실의 대출 규정을 확인하려면 어떤 도구를 써야 하나요?"

def first_call(tools: list) -> dict:
    reply = model.bind_tools(tools, tool_choice="required").invoke(question)
    usage = reply.usage_metadata
    assert usage is not None and reply.tool_calls
    return {
        "input_tokens": usage["input_tokens"],
        "output_tokens": usage["output_tokens"],
        "selected": reply.tool_calls[0]["name"],
    }

measurements = {
    "direct_1": first_call([direct_lookup]),
    "mcp_1": first_call(small_tools),
    "mcp_5": first_call(large_tools),
}
for condition, result in measurements.items():
    print(condition, result)
assert measurements["mcp_5"]["input_tokens"] > measurements["mcp_1"]["input_tokens"]

direct_1 {'input_tokens': 158, 'output_tokens': 18, 'selected': 'direct_lookup'}
mcp_1 {'input_tokens': 158, 'output_tokens': 18, 'selected': 'lookup_policy'}
mcp_5 {'input_tokens': 316, 'output_tokens': 31, 'selected': 'lookup_policy'}


## 비용 계산과 해석 범위

비용 계산과 해석 범위에 필요한 값과 결과를 확인합니다.

In [6]:
# 비용 계산과 해석 범위에 필요한 값을 준비하고 결과를 확인합니다.
INPUT_USD_PER_MTOK = 0.75
OUTPUT_USD_PER_MTOK = 4.50

for result in measurements.values():
    result["estimated_cost_usd"] = round(
        result["input_tokens"] / 1_000_000 * INPUT_USD_PER_MTOK
        + result["output_tokens"] / 1_000_000 * OUTPUT_USD_PER_MTOK,
        8,
    )
total_cost = sum(item["estimated_cost_usd"] for item in measurements.values())
print("measurements:", measurements)
print("total_estimated_cost_usd:", round(total_cost, 8))
print("MCP 전송 바이트는 이 토큰 수에 포함되지 않습니다.")
assert total_cost < 1.0

measurements: {'direct_1': {'input_tokens': 158, 'output_tokens': 18, 'selected': 'direct_lookup', 'estimated_cost_usd': 0.0001995}, 'mcp_1': {'input_tokens': 158, 'output_tokens': 18, 'selected': 'lookup_policy', 'estimated_cost_usd': 0.0001995}, 'mcp_5': {'input_tokens': 316, 'output_tokens': 31, 'selected': 'lookup_policy', 'estimated_cost_usd': 0.0003765}}
total_estimated_cost_usd: 0.0007755
MCP 전송 바이트는 이 토큰 수에 포함되지 않습니다.
